# Gradio UI Testing - Q&A System V2

Quick prototyping notebook for testing the Gradio interface with the refactored pipeline.

In [1]:
# Setup - adjust paths as needed
import sys
sys.path.append('..')

import gradio as gr
from qanda_module.config import setup_system
from qanda_module.retrieval_lean import process_question_with_style
import logging

# Enable logging to see what's happening
logging.basicConfig(level=logging.INFO)

print("📦 Imports successful!")

📦 Imports successful!


In [2]:
# Setup system - this should work from your previous testing
print("🚀 Setting up system...")

helpers, qa_chain, config = setup_system("../config.yaml")

print("✅ System setup complete!")
print(f"Model: {config.chat_model_name}")
print(f"Temperature: {config.temperature}")
print(f"Debug: {config.debug_enabled}")

🚀 Setting up system...


INFO:qanda_module.database_clean:Loading existing database...
INFO:qanda_module.database_clean:dim_panellist: 2180 records
INFO:qanda_module.database_clean:dim_question: 4032 records
INFO:qanda_module.database_clean:dim_episode: 470 records
INFO:qanda_module.database_clean:fact_responses: 118504 records
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


Loaded 1094 panelist profile URLs
Total panellist responses: 65031
Panellist responses with missing speaker_id: 0
✅ System setup complete!
Model: gpt-4o-mini
Temperature: 0.3
Debug: False


In [3]:
# API Key setup
import os
from dotenv import load_dotenv

# Try to load from .env file
load_dotenv()


False

In [4]:
# Test the basic pipeline first (without UI)
print("🧪 Testing basic pipeline...")

test_question = "What did panelists say about climate change?"
print(f"Question: {test_question}")

try:
    result = process_question_with_style(
        helpers=helpers,
        qa_chain=qa_chain,
        query=test_question,
        k=10,  # Small number for testing
        style="Concise",
        config=config
    )
    
    print("✅ Pipeline test successful!")
    print(f"Answer length: {len(result[0])} characters")
    print(f"Status: {result[2]}")
    print(f"\nFirst 200 chars of response:\n{result[0][:200]}...")
    
except Exception as e:
    print(f"❌ Pipeline test failed: {e}")
    import traceback
    traceback.print_exc()

🧪 Testing basic pipeline...
Question: What did panelists say about climate change?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


✅ Pipeline test successful!
Answer length: 610 characters
Status: ✅ Completed!

First 200 chars of response:
The panelists discussed the distinction between scientific evidence and personal experience regarding climate change. One audience member highlighted that only one scientist was on the panel, suggesti...


In [5]:
# Define the question handler function
def handle_question(question, k_value, style):
    """Handle user question and return response."""
    if not question.strip():
        return "Please enter a question.", "", "⚠️ No question provided"
    
    print(f"Processing: {question[:50]}...")  # Debug print
    
    try:
        result = process_question_with_style(
            helpers=helpers,
            qa_chain=qa_chain, 
            query=question,
            k=k_value,
            style=style,
            config=config
        )
        
        # result = (formatted_answer, sources, status, pdf_output, tech_info)
        return result[0], result[1], result[2]
        
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(f"❌ Error in handler: {error_msg}")
        return error_msg, "", f"❌ {error_msg}"

print("✅ Handler function defined")

✅ Handler function defined


In [6]:
# Create the Gradio interface
print("🎨 Creating Gradio interface...")

with gr.Blocks(
    title="Q&A System V2 - Test",
    theme=gr.themes.Soft()
) as demo:
    
    gr.Markdown("# 🧠 Q&A System V2 - Pipeline Test")
    gr.Markdown("Testing the refactored pipeline with basic Gradio interface.")
    
    with gr.Row():
        with gr.Column(scale=2):
            # Question input
            question = gr.Textbox(
                label="Your Question",
                placeholder="e.g., What did panelists say about climate change?",
                lines=3
            )
            
            # Sample questions
            sample_questions = gr.Dropdown(
                label="Sample Questions",
                choices=[
                    "What did panelists say about climate change?",
                    "How do panelists view immigration policy?", 
                    "What are different perspectives on the economy?",
                    "Which topics generated the most debate?",
                    "What did panelists say about housing affordability?",
                ],
                interactive=True
            )
            
            # Controls
            with gr.Row():
                k_slider = gr.Slider(
                    minimum=5,
                    maximum=50, 
                    value=15,  # Start small for testing
                    step=5,
                    label="Documents to retrieve (k)"
                )
                
                style_radio = gr.Radio(
                    label="Response Style",
                    choices=["Concise", "Balanced", "Detailed"],
                    value="Balanced"
                )
            
            # Buttons
            with gr.Row():
                submit_btn = gr.Button("🔍 Ask Question", variant="primary")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary")
        
        with gr.Column(scale=1):
            # Status
            gr.Markdown("### System Status")
            status_display = gr.Textbox(
                label="Status",
                value="Ready to test!",
                interactive=False
            )
            
            # Config info
            gr.Markdown(f"""
            **Configuration:**
            - Model: {config.chat_model_name}
            - Temperature: {config.temperature}
            - Debug: {config.debug_enabled}
            """)
    
    # Results
    gr.Markdown("### 📝 Response")
    answer_display = gr.Markdown(
        value="*Ask a question to see the response here.*"
    )
    
    gr.Markdown("### 📚 Sources") 
    sources_display = gr.Markdown(
        value="*Sources will appear here.*"
    )
    
    # Event handlers
    def set_sample_question(sample):
        return sample if sample else ""
    
    def clear_inputs():
        return "", "*Ask a question to see the response here.*", "Cleared - ready for next question"
    
    # Wire up events
    sample_questions.change(
        fn=set_sample_question,
        inputs=[sample_questions],
        outputs=[question]
    )
    
    submit_btn.click(
        fn=handle_question,
        inputs=[question, k_slider, style_radio],
        outputs=[answer_display, sources_display, status_display]
    )
    
    clear_btn.click(
        fn=clear_inputs,
        inputs=[],
        outputs=[question, answer_display, status_display]
    )

print("✅ Gradio interface created!")

🎨 Creating Gradio interface...
✅ Gradio interface created!


In [7]:
# Launch the interface
print("🌐 Launching Gradio interface...")

demo.launch(
    share=False,  # Set to True for public link
    server_port=7860,
    show_error=True,
    debug=True  # Shows more info in console
)

🌐 Launching Gradio interface...


INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Processing: How do panelists view immigration policy?...


...

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Processing: How do panelists view immigration policy?...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Keyboard interruption in main thread... closing server.


In [ ]:
# If you need to stop the demo
demo.close()

## 🧪 Quick Tests

Use the cells below for rapid testing and debugging:

In [ ]:
# Quick test different parameters
test_questions = [
    "What did panelists say about climate change?",
    "How do panelists view immigration?",
    "What are the different economic perspectives?"
]

for i, q in enumerate(test_questions[:1]):  # Test just first one
    print(f"\n=== Test {i+1}: {q} ===")
    result = handle_question(q, 10, "Concise")
    print(f"Status: {result[2]}")
    print(f"Answer (first 100 chars): {result[0][:100]}...")

In [ ]:
# Debug: Check what's in your helpers object
print("Helpers object attributes:")
print([attr for attr in dir(helpers) if not attr.startswith('_')])

print(f"\nDatabase info:")
print(f"- Episodes: {len(helpers.df_ep)}")
print(f"- Responses: {len(helpers.df_reply)}")
print(f"- Panelists: {len(helpers.df_guests)}")